# Hospital Revenue — Analysis Report

Notebook tự động sinh **5–10 key findings** từ kết quả phân tích đã có:
- `descriptive_stats.json` → thống kê mô tả
- `correlation_matrix.json` → tương quan
- `trend_analysis.json` → xu hướng
- `forecast.json` → dự báo

> **Yêu cầu:** Chạy `revenue_analysis.ipynb` và `forecasting.ipynb` trước khi chạy notebook này.
>
> Sau khi chạy xong, vào `http://127.0.0.1:5000/report` để xem báo cáo.

## Section 1 — Load dữ liệu từ JSON

In [1]:
import json
import datetime
from pathlib import Path

OUTPUT_DIR = Path("../backend/analysis_output")

def load_json(name):
    path = OUTPUT_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy {path}. Hãy chạy notebook tương ứng trước.")
    with open(path, encoding="utf-8") as f:
        return json.load(f)

desc  = load_json("descriptive_stats.json")
corr  = load_json("correlation_matrix.json")
trend = load_json("trend_analysis.json")
fc    = load_json("forecast.json")

print("Load thành công 4 file JSON.")
print(f"   descriptive_stats : {desc['generated_at'][:19]}")
print(f"   correlation_matrix: {corr['generated_at'][:19]}")
print(f"   trend_analysis    : {trend['generated_at'][:19]}")
print(f"   forecast          : {fc['generated_at'][:19]}")

Load thành công 4 file JSON.
   descriptive_stats : 2026-06-07T14:03:29
   correlation_matrix: 2026-06-07T14:03:29
   trend_analysis    : 2026-06-07T14:03:29
   forecast          : 2026-06-07T14:04:56


## Section 2 — Tự động sinh Key Findings

In [2]:
findings = []
total_rev = desc["overall"]["total"]

# ── Finding 1: Top department ────────────────────────────────────────
depts = sorted(desc["by_department"], key=lambda x: x["total"], reverse=True)
top   = depts[0]
share = top["total"] / total_rev * 100 if total_rev else 0
second_ratio = top["total"] / depts[1]["total"] if len(depts) > 1 and depts[1]["total"] else 1

findings.append({
    "id"            : 1,
    "title"         : f"{top['TEN_KHOAPHONG']} dẫn đầu doanh thu toàn viện",
    "metric"        : f"{share:.1f}% tổng doanh thu",
    "explanation"   : (
        f"Khoa {top['TEN_KHOAPHONG']} đóng góp {share:.1f}% tổng doanh thu "
        f"({top['total']:,.0f} VNĐ), cao hơn khoa đứng thứ 2 "
        f"({depts[1]['TEN_KHOAPHONG'] if len(depts) > 1 else '—'}) "
        f"{second_ratio:.1f} lần. Khoa này cũng có {top['unique_visits']:,.0f} lượt khám "
        f"với doanh thu bình quân {top['revenue_per_visit']:,.0f} VNĐ/lượt."
    ),
    "recommendation": (
        f"Đây là nguồn doanh thu cốt lõi. Ưu tiên duy trì chất lượng và năng lực "
        f"khoa {top['TEN_KHOAPHONG']}. Đồng thời xem xét nhân rộng mô hình vận hành "
        f"sang các khoa khác."
    ),
    "level"         : "positive",
})

# ── Finding 2: BHYT ratio ────────────────────────────────────────────
bhyt = desc["bhyt_ratio"]
if bhyt > 60:
    lvl = "warning"
    rec = ("Tỷ lệ BHYT cao đồng nghĩa với biên lợi nhuận thực tế thấp hơn. "
           "Cân nhắc phát triển dịch vụ kỹ thuật cao và theo yêu cầu để tăng doanh thu tự trả.")
elif bhyt < 30:
    lvl = "positive"
    rec = ("Tỷ lệ tự trả cao phản ánh biên lợi nhuận tốt. "
           "Duy trì chất lượng dịch vụ để giữ bệnh nhân tự nguyện.")
else:
    lvl = "info"
    rec = "Cơ cấu thanh toán cân bằng. Theo dõi định kỳ để phát hiện thay đổi bất thường."

findings.append({
    "id"            : 2,
    "title"         : f"Tỷ lệ BHYT chi trả ở mức {bhyt:.1f}%",
    "metric"        : f"BHYT {bhyt:.1f}% | Tự trả {100-bhyt:.1f}%",
    "explanation"   : (
        f"Bảo hiểm y tế chi trả {bhyt:.1f}% tổng doanh thu toàn viện. "
        f"Tỷ lệ này ảnh hưởng trực tiếp đến biên lợi nhuận thực tế "
        f"vì mức thanh toán BHYT thường thấp hơn giá dịch vụ niêm yết."
    ),
    "recommendation": rec,
    "level"         : lvl,
})

# ── Finding 3: Day of week pattern ──────────────────────────────────
dow       = sorted(desc["by_day_of_week"], key=lambda x: x["mean"], reverse=True)
best_day  = dow[0]
worst_day = dow[-1]
day_gap   = (best_day["mean"] - worst_day["mean"]) / best_day["mean"] * 100 if best_day["mean"] else 0
best_vi   = best_day.get("THU_VI", best_day.get("THU_TEN", "?"))
worst_vi  = worst_day.get("THU_VI", worst_day.get("THU_TEN", "?"))

findings.append({
    "id"            : 3,
    "title"         : f"{best_vi} là ngày có doanh thu cao nhất trong tuần",
    "metric"        : f"Cao hơn ngày thấp nhất {day_gap:.0f}%",
    "explanation"   : (
        f"{best_vi} có doanh thu trung bình {best_day['mean']:,.0f} VNĐ/giao dịch, "
        f"cao hơn {worst_vi} ({worst_day['mean']:,.0f} VNĐ) tới {day_gap:.0f}%. "
        f"Pattern này lặp lại đều cho thấy có hành vi bệnh nhân có tính mùa vụ theo tuần."
    ),
    "recommendation": (
        f"Tối ưu lịch nhân sự tập trung vào {best_vi}. "
        f"Cân nhắc chương trình khuyến khích đặt lịch vào {worst_vi} để cân bằng tải."
    ),
    "level"         : "info",
})

# ── Finding 4: Service group concentration ──────────────────────────
svcs      = sorted(desc["by_service_group"], key=lambda x: x["total"], reverse=True)
top_svc   = svcs[0]
svc_share = top_svc["total"] / total_rev * 100 if total_rev else 0

findings.append({
    "id"            : 4,
    "title"         : f"Nhóm '{top_svc['NHOM_DICHVU']}' chiếm tỷ trọng cao nhất",
    "metric"        : f"{svc_share:.1f}% tổng doanh thu",
    "explanation"   : (
        f"Nhóm dịch vụ '{top_svc['NHOM_DICHVU']}' đóng góp {svc_share:.1f}% tổng doanh thu "
        f"({top_svc['total']:,.0f} VNĐ). "
        f"{'Tỷ trọng quá cao tạo rủi ro tập trung.' if svc_share > 40 else 'Phân bổ dịch vụ tương đối đa dạng.'}"
    ),
    "recommendation": (
        "Đa dạng hóa danh mục dịch vụ để giảm phụ thuộc vào một nhóm duy nhất. "
        "Đầu tư phát triển các nhóm dịch vụ tiềm năng như chẩn đoán hình ảnh và xét nghiệm."
        if svc_share > 40 else
        "Tiếp tục duy trì danh mục dịch vụ đa dạng. Theo dõi xu hướng từng nhóm để nắm bắt cơ hội."
    ),
    "level"         : "warning" if svc_share > 40 else "info",
})

# ── Finding 5: Revenue per visit ────────────────────────────────────
rpv        = desc["revenue_per_visit"]
mean_tx    = desc["overall"]["mean"]
tx_per_visit = rpv / mean_tx if mean_tx else 1

findings.append({
    "id"            : 5,
    "title"         : f"Mỗi lượt khám tạo ra trung bình {rpv:,.0f} VNĐ",
    "metric"        : f"{tx_per_visit:.1f} dịch vụ/lượt khám",
    "explanation"   : (
        f"Doanh thu trung bình mỗi lượt khám là {rpv:,.0f} VNĐ, tương đương khoảng "
        f"{tx_per_visit:.1f} dịch vụ/lượt (so với đơn giá trung bình {mean_tx:,.0f} VNĐ/dịch vụ). "
        f"Chỉ số này phản ánh mức độ sử dụng dịch vụ kết hợp của bệnh nhân."
    ),
    "recommendation": (
        "Tăng revenue per visit bằng cách phát triển gói khám tổng hợp kết hợp "
        "nhiều dịch vụ (khám + xét nghiệm + chẩn đoán hình ảnh). "
        "Đây là đòn bẩy hiệu quả để tăng doanh thu mà không cần tăng số lượt khám."
    ),
    "level"         : "info",
})

# ── Finding 6: Correlation highlight ────────────────────────────────
labels = corr["department_correlation"]["labels"]
matrix = corr["department_correlation"]["matrix"]
max_corr, max_pair = -2, ("", "")
min_corr, min_pair = 2,  ("", "")

for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        v = matrix[i][j]
        if v > max_corr:
            max_corr, max_pair = v, (labels[i], labels[j])
        if v < min_corr:
            min_corr, min_pair = v, (labels[i], labels[j])

if max_corr > 0.5:
    findings.append({
        "id"            : 6,
        "title"         : f"{max_pair[0]} và {max_pair[1]} biến động cùng chiều mạnh",
        "metric"        : f"r = {max_corr:.2f}",
        "explanation"   : (
            f"Hệ số Pearson correlation giữa {max_pair[0]} và {max_pair[1]} là {max_corr:.2f}, "
            f"cho thấy doanh thu của hai khoa này tăng giảm rất đồng bộ. "
            f"Nguyên nhân có thể do chung nguồn bệnh nhân, dịch vụ bổ trợ nhau, "
            f"hoặc chịu ảnh hưởng từ cùng yếu tố ngoại cảnh (mùa bệnh, lễ tết)."
        ),
        "recommendation": (
            f"Phân tích luồng bệnh nhân giữa {max_pair[0]} và {max_pair[1]}. "
            "Nếu là dịch vụ bổ trợ, thiết kế gói liên khoa để tăng tỷ lệ chuyển tuyến nội bộ "
            "và tối ưu doanh thu trên mỗi bệnh nhân."
        ),
        "level"         : "info",
    })

# ── Finding 7: Monthly growth trend ─────────────────────────────────
monthly = trend["monthly_revenue"]
growth_rates = [
    m["growth_rate"] for m in monthly
    if m.get("growth_rate") is not None and m["growth_rate"] != 0
]

if growth_rates:
    avg_growth   = sum(growth_rates) / len(growth_rates)
    pos_months   = sum(1 for g in growth_rates if g > 0)
    total_months = len(growth_rates)
    lvl          = "positive" if avg_growth > 0 else "warning"

    findings.append({
        "id"            : 7,
        "title"         : f"Tốc độ tăng trưởng trung bình {avg_growth:+.1f}%/tháng",
        "metric"        : f"{pos_months}/{total_months} tháng tăng trưởng dương",
        "explanation"   : (
            f"Trong {total_months} tháng phân tích, có {pos_months} tháng ghi nhận tăng trưởng dương. "
            f"Tốc độ tăng trưởng trung bình đạt {avg_growth:+.1f}%/tháng. "
            f"{'Xu hướng tích cực, bệnh viện đang trên đà phát triển.' if avg_growth > 0 else 'Xu hướng cần cải thiện.'}"
        ),
        "recommendation": (
            "Duy trì đà tăng trưởng bằng cách mở rộng năng lực khám và phát triển dịch vụ mới."
            if avg_growth > 0 else
            "Rà soát nguyên nhân tăng trưởng âm: chất lượng dịch vụ, cạnh tranh, hay yếu tố mùa vụ. "
            "Xây dựng kế hoạch cải thiện doanh thu ngắn hạn."
        ),
        "level"         : lvl,
    })

# ── Finding 8: Forecast ──────────────────────────────────────────────
fc_mf       = fc["monthly_forecast"]
fc_growth   = fc_mf["growth_vs_last_30d"]
lvl         = "positive" if fc_growth > 0 else "warning"

findings.append({
    "id"            : 8,
    "title"         : f"Dự báo doanh thu tháng tới {'tăng' if fc_growth > 0 else 'giảm'} {abs(fc_growth):.1f}%",
    "metric"        : f"{fc_mf['predicted_total']:,.0f} VNĐ",
    "explanation"   : (
        f"Model {fc['model']} dự báo tổng doanh thu 30 ngày tới đạt "
        f"{fc_mf['predicted_total']:,.0f} VNĐ — "
        f"{'tăng' if fc_growth > 0 else 'giảm'} {abs(fc_growth):.1f}% so với 30 ngày vừa qua. "
        f"Khoảng tin cậy 95%: {fc_mf['lower_95']:,.0f} → {fc_mf['upper_95']:,.0f} VNĐ."
    ),
    "recommendation": (
        "Chuẩn bị nhân lực và vật tư phù hợp với mức tăng dự kiến. "
        "Theo dõi sát doanh thu thực tế hàng ngày để phát hiện lệch so với dự báo sớm."
        if fc_growth > 0 else
        "Xem xét các biện pháp kích cầu: chương trình khuyến mãi, gói khám, truyền thông. "
        "Tối ưu lịch bác sĩ để giảm tỷ lệ hủy lịch và tăng tỷ lệ tái khám."
    ),
    "level"         : lvl,
})

# ── Summary ──────────────────────────────────────────────────────────
n_pos  = sum(1 for f in findings if f["level"] == "positive")
n_warn = sum(1 for f in findings if f["level"] == "warning")
n_info = len(findings) - n_pos - n_warn

executive_summary = (
    f"Phân tích {desc['overall']['count']:,} giao dịch với tổng doanh thu "
    f"{total_rev:,.0f} VNĐ từ {desc['unique_patients']:,} bệnh nhân. "
    f"Dự báo 30 ngày tới "
    f"{'tăng trưởng ' + str(abs(round(fc_growth, 1))) + '%' if fc_growth > 0 else 'giảm ' + str(abs(round(fc_growth, 1))) + '%'}. "
    f"Tổng hợp {len(findings)} phát hiện: {n_pos} tích cực, {n_warn} cần theo dõi, {n_info} thông tin."
)

print(executive_summary)
print()
for f in findings:
    icon = "✅" if f["level"] == "positive" else "⚠️" if f["level"] == "warning" else "ℹ️"
    print(f"{icon} #{f['id']} {f['title']} — {f['metric']}")

Phân tích 20,000 giao dịch với tổng doanh thu 44,972,052,738 VNĐ từ 1,044 bệnh nhân. Dự báo 30 ngày tới tăng trưởng 37.7%. Tổng hợp 7 phát hiện: 2 tích cực, 2 cần theo dõi, 3 thông tin.

✅ #1 Khoa Sản dẫn đầu doanh thu toàn viện — 10.8% tổng doanh thu
ℹ️ #2 Tỷ lệ BHYT chi trả ở mức 35.9% — BHYT 35.9% | Tự trả 64.1%
ℹ️ #3 Thứ Hai là ngày có doanh thu cao nhất trong tuần — Cao hơn ngày thấp nhất 15%
⚠️ #4 Nhóm 'Phẫu thuật / thủ thuật' chiếm tỷ trọng cao nhất — 75.1% tổng doanh thu
ℹ️ #5 Mỗi lượt khám tạo ra trung bình 2,258,200 VNĐ — 1.0 dịch vụ/lượt khám
⚠️ #7 Tốc độ tăng trưởng trung bình -3.5%/tháng — 6/11 tháng tăng trưởng dương
✅ #8 Dự báo doanh thu tháng tới tăng 37.7% — 6,623,535,850 VNĐ


## Section 3 — Export report.json cho Dashboard

In [3]:
generated_at = datetime.datetime.now().isoformat()

report_output = {
    "generated_at"      : generated_at,
    "executive_summary" : executive_summary,
    "total_findings"    : len(findings),
    "n_positive"        : n_pos,
    "n_warning"         : n_warn,
    "n_info"            : n_info,
    "findings"          : findings,
    "data_sources": {
        "descriptive_stats" : desc["generated_at"],
        "correlation_matrix": corr["generated_at"],
        "trend_analysis"    : trend["generated_at"],
        "forecast"          : fc["generated_at"],
    },
}

out = OUTPUT_DIR / "report.json"
out.write_text(json.dumps(report_output, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"{out}")
print(f"\nExport hoàn tất lúc {generated_at[:19]}")
print(f"   {len(findings)} findings → http://127.0.0.1:5000/report")

../backend/analysis_output/report.json

Export hoàn tất lúc 2026-06-07T14:06:57
   7 findings → http://127.0.0.1:5000/report
